# 03 — Nấc 3: SFT trên Qwen3-8B

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/03_sft_qwen3.ipynb)

**Cần GPU.** Phần A ~40–60 phút, phần B ~1.5–2 giờ trên L4.

## Vì sao KHÔNG fine-tune thẳng trên nhãn vàng

Lần chạy trước đã thử đúng cách đó (Stage 2: LoRA Mistral-7B trên gold thô) và **thất bại**:

| Step | Train loss | Val loss |
|---|---|---|
| 50 | 0.7277 | 0.9122 |
| 150 | 0.3678 | 0.9268 |
| 250 | 0.2384 | 0.9473 |

Train loss giảm đều, val loss tăng rồi đứng — overfit kinh điển. Có hai nguyên nhân:
nhãn vàng có 5–10 % không nhất quán, và **đích huấn luyện chỉ là program trần, không có
chuỗi suy luận**, nên model không học được cách suy luận mà chỉ học thuộc.

## Cách làm ở nấc này: rejection sampling (self-distillation)

1. Chạy chính Qwen3-8B (chưa fine-tune) với **prompt của nấc 2** trên tập train.
2. Giữ lại những mẫu model làm **đúng** (PA hoặc EA).
3. Dùng chính lời giải đó — *đã có đầy đủ chuỗi suy luận tiếng Việt* — làm đích huấn luyện.

Ba ưu điểm so với cách SFT thẳng trên gold:

* Đích huấn luyện **có chuỗi suy luận**, đúng văn phong model vốn sinh ra được → tránh
  đúng nguyên nhân overfit đã nêu.
* **Không cần API ngoài.** Cách cũ phải dùng Gemini-2.5-flash sửa 600 mẫu, tức là rơi sang
  thiết lập *unconstrained*. Cách này vẫn nằm trong **constrained-resource**.
* Mẫu có nhãn nhiễu phần lớn tự rơi ra: model làm "đúng theo gold" rất khó khi gold sai.
  Ta còn lọc thêm bằng `data.is_noisy_gold()`.

**Hạn chế phải nêu khi báo cáo:** chỉ học được từ những gì model *đã* làm được, nên khó dạy
kiểu bài model chưa bao giờ giải đúng. Tuỳ chọn `ADD_GOLD_FALLBACK` bù một phần nhưng bật
nó là quay lại đúng rủi ro overfit của cách cũ — mặc định tắt.

## Cấu trúc notebook

* **Phần A** (§1–§5): dựng dữ liệu SFT. Cần vLLM.
* **⚠ RESTART RUNTIME** giữa hai phần — vLLM giữ VRAM rất chặt, không nhả đủ cho training.
* **Phần B** (§6–§10): huấn luyện LoRA, rồi chấm trên test.

## Tham số LoRA

Giữ nguyên `reference/original_notebooks/finetune_phi4.ipynb`: r=16, alpha=32, dropout=0,
7 target modules, `use_gradient_checkpointing="unsloth"`, `paged_adamw_8bit`,
lr=2e-4, cosine, warmup 0.1, weight_decay 0.05, early stopping patience 3.
Batch được tự chỉnh theo VRAM nhưng **batch hiệu dụng giữ nguyên 16** để lr 2e-4 còn hợp lệ.

# PHẦN A — Dựng dữ liệu SFT

## §1. Môi trường

In [ ]:
# Khối cài đặt giữ NGUYÊN của reference/original_notebooks/inference_with_difference_models.ipynb
# để môi trường khớp với lần chạy tham chiếu.
#
# ⏱ Mất 12–20 PHÚT, gần hết thời gian nằm ở `pip install vllm` (wheel rất nặng).
#   Cố ý KHÔNG giấu output (không dùng %%capture) để nhìn được nó còn sống:
#   còn thấy dòng mới hiện ra là đang chạy. Quá ~30 phút mà đứng im một chỗ mới là treo.
#   Chỉ chạy một lần mỗi session; restart runtime KHÔNG phải cài lại.
import os, re, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install vllm
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô."""
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model (bản gốc, chưa fine-tune)

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1, max_tokens=3000
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Trên Kaggle chọn 'GPU T4 x2', đừng chọn P100.")

TEMPERATURE, MAX_TOKENS = 0.1, 3000        # giữ đúng mốc tham chiếu
REPETITION_PENALTY = 1.0
BATCH_SIZE = 500                           # như notebook cũ; giảm nếu OOM

if _VRAM < 18:                             # T4 15GB
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 8192, 0.90, 16
    MAX_TOKENS, BATCH_SIZE = 1536, 32
elif _VRAM < 30:                           # L4 24GB
    # 13500 chứ không phải 12000: để ngân sách prompt (13500-3000=10500) vượt xa
    # prompt bước 2 dài nhất (~9k token) → L4 KHÔNG phải cắt ngữ cảnh, nhờ vậy
    # kết quả trên L4 và A100 so sánh trực tiếp được với nhau.
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 13500, 0.88, 24
    BATCH_SIZE = 128
elif _VRAM < 60:                           # A100 40GB — như notebook gốc
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 64
else:                                      # A100 80GB — còn dư, tăng song song
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 128
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"batch={BATCH_SIZE}")
if _VRAM < 18:
    print("[CFG] ⚠ T4: đã hạ max_tokens xuống 1536 → kết quả KHÔNG so trực tiếp "
          "được với máy dùng 3000. Nên chạy trên L4.")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"[MODEL] Đang tải {MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    dtype                  = DTYPE,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = True,
    gpu_memory_utilization = GPU_MEM_UTIL,
    max_num_seqs           = MAX_NUM_SEQS,
)

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        outs.extend(o.outputs[0].text for o in model.fast_generate(chunk, **kw))
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s" + " "*16)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng | VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(REPO_DIR, tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False
if PROMPT_LEVEL == "selfeval":
    PROMPT_LEVEL = "engineered"

print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL}")
print(f"         plain={len(prompt_kit.PLAIN_SYSTEM_PROMPT)} ký tự | "
      f"engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)} | "
      f"self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
_prev = "Phân tích chi tiết. " * 150 + "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

## §3. Chọn tập train để sinh dữ liệu

`SFT_TRAIN_SUBSET` là số mẫu train đem ra chạy sinh. Vì chỉ giữ lại mẫu model làm đúng
(~50–60 % theo kết quả nấc 2), lấy 2000 mẫu sẽ cho khoảng 1000–1200 mẫu huấn luyện —
cùng cỡ với 2237 mẫu mà lần chạy trước dùng cho Phi-4.

Mẫu được lấy **phân tầng theo số phép toán** để dữ liệu huấn luyện không chỉ toàn bài 1 bước.

In [ ]:
SFT_TRAIN_SUBSET = 2000        # đặt None để dùng cả 2993 mẫu (lâu hơn ~50%)
DROP_NOISY_GOLD  = True        # bỏ mẫu multiply(#n,100) và gold lỗi cú pháp
ACCEPT           = "pa_or_ea"  # 'pa' chặt nhất | 'ea' | 'pa_or_ea'
ADD_GOLD_FALLBACK = False      # bật = quay lại rủi ro overfit của cách cũ

sft_train = data.stratified_sample(train_all, SFT_TRAIN_SUBSET, seed=RANDOM_SEED)
print(f"[SFT] lấy {len(sft_train)} mẫu train")
print("  phân tầng:", dict(sorted(Counter(min(dsl.n_ops(s['qa']['program']), 5)
                                          for s in sft_train).items())))
print("  nguồn    :", {k: len(v) for k, v in data.split_by_source(sft_train).items()})
_nz = sum(1 for s in sft_train if data.is_noisy_gold(s))
print(f"  nhãn nhiễu: {_nz} ({_nz/len(sft_train)*100:.1f}%)"
      f"{' → sẽ bị lọc' if DROP_NOISY_GOLD else ' → GIỮ LẠI (không khuyến nghị)'}")

## §4. Sinh lời giải trên tập train

Chạy đúng pipeline của nấc 2 (prompt có cấu trúc, một bước) trên tập train.
Đây là bước tốn thời gian nhất của phần A.

In [ ]:
# (phần đọc/ghi đã nằm ở cell đầu)

In [ ]:
_t0 = time.time()
train_rows = pipeline.run_pipeline(
    sft_train, prompt_kit, generate,
    prompt_level="engineered", use_selfeval=False,
    sp_step1=SAMPLING, desc="sinh-train", keep_raw=True)
_m_train = pipeline.summarize(train_rows, "train (để dựng SFT)")
print(f"\n  Thời gian: {(time.time()-_t0)/60:.1f} phút")
pipeline.print_summary(_m_train)
print(f"\n  → Tỉ lệ làm đúng trên train ≈ tỉ lệ mẫu sẽ vào được dữ liệu SFT.")

## §5. Lọc và ghi dữ liệu SFT

Cell dưới in rõ **mỗi mẫu bị loại vì lý do gì** — để kiểm chứng rằng việc lọc là có cơ sở
chứ không phải cắt bừa dữ liệu khó.

In [ ]:
records, stats_build = sft.build_sft_records(
    train_rows, sft_train, prompt_kit,
    level="engineered", accept=ACCEPT,
    add_gold_fallback=ADD_GOLD_FALLBACK, drop_noisy_gold=DROP_NOISY_GOLD)

print(f"{'═'*62}\n  DỰNG DỮ LIỆU SFT\n{'═'*62}")
for k, v in sorted(stats_build.items(), key=lambda x: -x[1]):
    print(f"  {k:<28}{v:>6}")
print(f"\n  → giữ lại {len(records)} mẫu huấn luyện")

st = sft.sft_data_stats(records)
print(f"\n  Phân bố theo số phép toán: {st['theo_so_phep']}")
print(f"  Nguồn đích               : {st['theo_nguon']}")
print(f"  Độ dài (ký tự) p50/p95/max: {st['ky_tu_p50']} / {st['ky_tu_p95']} / {st['ky_tu_max']}")

assert len(records) >= 200, (
    f"Chỉ {len(records)} mẫu — quá ít để fine-tune. Tăng SFT_TRAIN_SUBSET, "
    f"hoặc đổi ACCEPT='ea', hoặc kiểm tra lại nấc 2.")

SFT_JSONL = os.path.join(OUTPUT_DIR, "sft_data", f"qwen3_sft_{STAMP}.jsonl")
sft.write_jsonl(records, SFT_JSONL)
with open(os.path.join(OUTPUT_DIR, "sft_data", "latest.txt"), "w") as f:
    f.write(SFT_JSONL)
print(f"\n[SAVE] {SFT_JSONL}")

# Hồ sơ dựng dữ liệu: bao nhiêu mẫu bị loại vì lý do gì, cấu hình nào sinh ra nó.
_build = os.path.join(OUTPUT_DIR, "sft_data", f"qwen3_sft_{STAMP}_build.json")
json.dump({"stamp": STAMP, "jsonl": SFT_JSONL, "n_records": len(records),
           "loc": stats_build, "phan_bo": st,
           "cau_hinh": {"SFT_TRAIN_SUBSET": SFT_TRAIN_SUBSET, "ACCEPT": ACCEPT,
                        "DROP_NOISY_GOLD": DROP_NOISY_GOLD,
                        "ADD_GOLD_FALLBACK": ADD_GOLD_FALLBACK,
                        "n_train_da_sinh": len(train_rows)},
           "metrics_tren_train": pipeline.summarize(train_rows, "train (sinh de loc)"),
           "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
           "env": run_env()},
          open(_build, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print(f"[SAVE] {_build}   ← hồ sơ lọc dữ liệu")

print(f"\n{'═'*62}\n  MỘT MẪU HUẤN LUYỆN\n{'═'*62}")
_ex = records[0]["messages"]
print(f"  [system] {len(_ex[0]['content'])} ký tự (prompt có cấu trúc, cắt bớt khi in)")
print(f"  [user]   {_ex[1]['content'][:300]}...")
print(f"  [assistant] ↓ đây là thứ model sẽ học sinh ra")
print(_ex[2]['content'][:900])

---

# ⚠ RESTART RUNTIME TẠI ĐÂY

**Runtime → Restart session**, rồi chạy tiếp từ §6.

Lý do: engine vLLM giữ VRAM rất chặt và không nhả đủ cho việc huấn luyện trong cùng phiên.
Dữ liệu SFT đã ghi ra Drive nên phần B đọc lại được, không mất gì.

---

# PHẦN B — Huấn luyện LoRA

## §6. Môi trường (chạy lại sau restart)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô."""
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# Tìm file dữ liệu SFT vừa dựng ở phần A
_latest = os.path.join(OUTPUT_DIR, "sft_data", "latest.txt")
if os.path.exists(_latest):
    SFT_JSONL = open(_latest).read().strip()
else:
    _cands = sorted(__import__("glob").glob(
        os.path.join(OUTPUT_DIR, "sft_data", "qwen3_sft_*.jsonl")))
    assert _cands, "Không thấy dữ liệu SFT — chạy phần A trước."
    SFT_JSONL = _cands[-1]

_n = sum(1 for _ in open(SFT_JSONL, encoding="utf-8"))
print(f"[SFT] dữ liệu: {SFT_JSONL}")
print(f"[SFT] {_n} mẫu")

## §7. Nạp model ở chế độ huấn luyện

Khác phần A ở hai chỗ: **không** bật `fast_inference` (đó là engine suy luận), và gắn thêm
LoRA adapter qua `get_peft_model`.

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"
_VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
_CC = torch.cuda.get_device_capability(0)
BF16 = _CC[0] >= 8                       # T4 (SM75) không có bfloat16

# Đích huấn luyện là lời giải bước 1 nên ngắn hơn nhiều so với SFT bước 2 của cách cũ
# (notebook cũ phải để 12000). 8192 là đủ và nhẹ hơn hẳn.
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(model, **sft.lora_config())

print(f"[GPU] {torch.cuda.get_device_name(0)} | {_VRAM:.1f} GB | bf16={BF16}")
print(f"[LoRA] {sft.lora_config()}")

## §8. Chuẩn bị dataset

Chia 90/10 train/val, áp chat template của Qwen3, và **lọc bỏ mẫu vượt `max_seq_length`**
(giống bước lọc `full_tokens <= MAX_TOKEN` của notebook cũ).

In [ ]:
from datasets import load_dataset

full_ds = load_dataset("json", data_files=SFT_JSONL, split="train")
split = full_ds.train_test_split(test_size=0.1, seed=3407)
train_ds, val_ds = split["train"], split["test"]

def _format(ex):
    # Dùng chat template có sẵn của Qwen3 — KHÔNG ghi đè như notebook cũ làm với phi-3
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False,
                                                  add_generation_prompt=False)}

train_ds = train_ds.map(_format)
val_ds = val_ds.map(_format)

def _ntok(ex):
    return {"n_tokens": len(tokenizer(ex["text"]).input_ids)}

train_ds = train_ds.map(_ntok)
val_ds = val_ds.map(_ntok)

_before = (len(train_ds), len(val_ds))
train_ds = train_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH)
val_ds = val_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH)
print(f"[DATA] train {_before[0]} → {len(train_ds)} | val {_before[1]} → {len(val_ds)}"
      f"  (lọc mẫu > {MAX_SEQ_LENGTH} token)")

import numpy as _np
_t = _np.array(train_ds["n_tokens"])
print(f"[DATA] token: p50={int(_np.percentile(_t,50))} p95={int(_np.percentile(_t,95))} "
      f"max={_t.max()}")
assert len(train_ds) >= 100, "Quá ít mẫu sau khi lọc."

## §9. Huấn luyện

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, TrainerCallback

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "sft_adapter_qwen3")
CKPT_DIR = os.path.join(OUTPUT_DIR, "sft_checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

cfg = sft.training_config(_VRAM, len(train_ds), epochs=3, output_dir=CKPT_DIR, bf16=BF16)
meta = cfg.pop("_meta")
print(f"[TRAIN] batch/thiết bị={cfg['per_device_train_batch_size']} × "
      f"accum={cfg['gradient_accumulation_steps']} → batch hiệu dụng={meta['effective_batch']}")
print(f"[TRAIN] {meta['steps_per_epoch']} step/epoch × 3 epoch = {meta['total_steps']} step "
      f"| eval mỗi {meta['eval_every']} step")

class ClearCache(TrainerCallback):
    def on_evaluate(self, args, state, control, **kw):
        gc.collect(); torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=TrainingArguments(**cfg),
    dataset_text_field="text", packing=False, max_seq_length=MAX_SEQ_LENGTH,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3), ClearCache()],
)

# Tiếp tục từ checkpoint nếu Colab đã ngắt giữa chừng — nhưng CHỈ khi checkpoint đó
# thuộc đúng lần huấn luyện này. Nối tiếp checkpoint của một bộ dữ liệu khác là hỏng
# âm thầm: sai dữ liệu, sai số step, adapter ra không phải cái mình tưởng.
_sig = {"sft_data": os.path.basename(SFT_JSONL), "n_train": len(train_ds),
        "effective_batch": meta["effective_batch"], "total_steps": meta["total_steps"]}
_sig_path = os.path.join(CKPT_DIR, "run_signature.json")
try:
    _old_sig = json.load(open(_sig_path, encoding="utf-8"))
except Exception:
    _old_sig = None

_ck = sorted([d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
             key=lambda x: int(x.split("-")[-1]))
if _ck and _old_sig != _sig:
    try:
        _moved = f"{CKPT_DIR}_cu_{datetime.now().strftime('%Y%m%d_%H%M')}"
        os.rename(CKPT_DIR, _moved)
        os.makedirs(CKPT_DIR, exist_ok=True)
        print(f"[TRAIN] ⚠ checkpoint cũ thuộc lần chạy khác → đã dời sang {_moved}")
    except OSError:
        print("[TRAIN] ⚠ checkpoint cũ thuộc lần chạy khác → bỏ qua, không nối tiếp")
    _ck = []
    print(f"[TRAIN]   (cũ: {_old_sig} | nay: {_sig})")
try:
    json.dump(_sig, open(_sig_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

_resume = os.path.join(CKPT_DIR, _ck[-1]) if _ck else None
print(f"[TRAIN] {'tiếp tục từ ' + _resume if _resume else 'bắt đầu từ đầu'}")

trainer_stats = trainer.train(resume_from_checkpoint=_resume)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n[SAVE] adapter → {ADAPTER_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
tr = logs[logs.get("loss").notna()] if "loss" in logs else pd.DataFrame()
ev = logs[logs.get("eval_loss").notna()] if "eval_loss" in logs else pd.DataFrame()

plt.figure(figsize=(9, 5))
if not tr.empty:
    plt.plot(tr["step"], tr["loss"], label="train loss")
if not ev.empty:
    plt.plot(ev["step"], ev["eval_loss"], marker="o", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("SFT Qwen3-8B — rejection sampling data")
_p = os.path.join(OUTPUT_DIR, f"sft_loss_{STAMP}.png")
plt.savefig(_p, dpi=150); plt.show()

if not ev.empty:
    _best = ev.loc[ev["eval_loss"].idxmin()]
    print(f"  val loss thấp nhất = {_best['eval_loss']:.4f} tại step {int(_best['step'])}")
    if ev["eval_loss"].iloc[-1] > ev["eval_loss"].min() * 1.05:
        print("  ⚠ val loss đã tăng trở lại → có dấu hiệu overfit; early stopping đã chặn.")
    else:
        print("  ✅ val loss không tăng ngược — khác hẳn đường cong SFT của cách cũ.")
print(f"[SAVE] {_p}")

# Lịch sử huấn luyện: đường cong loss + cấu hình, để soi lại mà không cần chạy lại.
_hist = os.path.join(OUTPUT_DIR, f"sft_history_{STAMP}.json")
json.dump({"stamp": STAMP, "log_history": trainer.state.log_history,
           "best_eval_loss": (float(ev["eval_loss"].min()) if not ev.empty else None),
           "best_step": (int(ev.loc[ev["eval_loss"].idxmin(), "step"])
                         if not ev.empty else None),
           "n_train": len(train_ds), "n_val": len(val_ds),
           "lora": sft.lora_config(), "training_args": cfg, "batch_meta": meta,
           "sft_data": SFT_JSONL,
           "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
           "env": run_env()},
          open(_hist, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print(f"[SAVE] {_hist}   ← đường cong loss + cấu hình huấn luyện")

---

# ⚠ RESTART RUNTIME TẠI ĐÂY (lần 2)

**Runtime → Restart session**, rồi chạy **từ cell #17 trở đi** (chính là §10 bên dưới).

Đừng chạy lại §6–§9: §9 là ô huấn luyện, chạy lại là train lại từ đầu mất 28 phút.
Phần C tự lo đủ mọi thứ nó cần — cài đặt, cấu hình, nạp model, nạp adapter.

Lý do restart: phiên hiện tại đang ở chế độ huấn luyện, cần quay lại engine vLLM để sinh.
Adapter đã lưu trên Drive nên không mất gì.

---

# PHẦN C — Chấm model đã SFT

## §10. Chấm trên tập test

Adapter được nạp vào engine vLLM qua `model.load_lora`, prompt giữ nguyên của nấc 2 — đúng
định dạng model vừa được huấn luyện.

In [ ]:
# Khối cài đặt giữ NGUYÊN của reference/original_notebooks/inference_with_difference_models.ipynb
# để môi trường khớp với lần chạy tham chiếu.
#
# ⏱ Mất 12–20 PHÚT, gần hết thời gian nằm ở `pip install vllm` (wheel rất nặng).
#   Cố ý KHÔNG giấu output (không dùng %%capture) để nhìn được nó còn sống:
#   còn thấy dòng mới hiện ra là đang chạy. Quá ~30 phút mà đứng im một chỗ mới là treo.
#   Chỉ chạy một lần mỗi session; restart runtime KHÔNG phải cài lại.
import os, re, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install vllm
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô."""
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1, max_tokens=3000
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Trên Kaggle chọn 'GPU T4 x2', đừng chọn P100.")

TEMPERATURE, MAX_TOKENS = 0.1, 3000        # giữ đúng mốc tham chiếu
REPETITION_PENALTY = 1.0
BATCH_SIZE = 500                           # như notebook cũ; giảm nếu OOM

if _VRAM < 18:                             # T4 15GB
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 8192, 0.90, 16
    MAX_TOKENS, BATCH_SIZE = 1536, 32
elif _VRAM < 30:                           # L4 24GB
    # 13500 chứ không phải 12000: để ngân sách prompt (13500-3000=10500) vượt xa
    # prompt bước 2 dài nhất (~9k token) → L4 KHÔNG phải cắt ngữ cảnh, nhờ vậy
    # kết quả trên L4 và A100 so sánh trực tiếp được với nhau.
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 13500, 0.88, 24
    BATCH_SIZE = 128
elif _VRAM < 60:                           # A100 40GB — như notebook gốc
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 64
else:                                      # A100 80GB — còn dư, tăng song song
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 128
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"batch={BATCH_SIZE}")
if _VRAM < 18:
    print("[CFG] ⚠ T4: đã hạ max_tokens xuống 1536 → kết quả KHÔNG so trực tiếp "
          "được với máy dùng 3000. Nên chạy trên L4.")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"[MODEL] Đang tải {MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    dtype                  = DTYPE,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = True,
    gpu_memory_utilization = GPU_MEM_UTIL,
    max_num_seqs           = MAX_NUM_SEQS,
)

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        outs.extend(o.outputs[0].text for o in model.fast_generate(chunk, **kw))
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s" + " "*16)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng | VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(REPO_DIR, tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False
if PROMPT_LEVEL == "selfeval":
    PROMPT_LEVEL = "engineered"

print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL}")
print(f"         plain={len(prompt_kit.PLAIN_SYSTEM_PROMPT)} ký tự | "
      f"engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)} | "
      f"self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
_prev = "Phân tích chi tiết. " * 150 + "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

In [ ]:
# (phần đọc/ghi đã nằm ở cell đầu)

In [ ]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "sft_adapter_qwen3")
assert os.path.isdir(ADAPTER_DIR), f"Không thấy adapter tại {ADAPTER_DIR} — chạy §9 trước."

LORA_REQUEST = model.load_lora(ADAPTER_DIR)
print(f"[LoRA] đã nạp adapter từ {ADAPTER_DIR}")
print("       generate() sẽ tự truyền lora_request vào fast_generate.")

In [ ]:
STAGE = "03_sft"
print(f"\n{'═'*74}\n  NẤC: {STAGE} | prompt={PROMPT_LEVEL} | self-eval={USE_SELFEVAL}"
      f" | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level=PROMPT_LEVEL, use_selfeval=USE_SELFEVAL,
    sp_step1=SAMPLING, sp_step2=SAMPLING, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics)
print(f"\n  Thời gian: {metrics['minutes']} phút")

### So với nấc 2

In [ ]:
_prev_rows = load_stage("02_prompt_eng")
if _prev_rows is None:
    print("[SO SÁNH] ⚠ chưa có kết quả nấc '02_prompt_eng' → bỏ qua phần kiểm định.")
    print("          Chạy notebook nấc trước rồi quay lại cell này.")
else:
    _m_prev = pipeline.summarize(_prev_rows, "02_prompt_eng")
    print(f"\n{'═'*84}\n  NẤC 3 vs NẤC 2 — giá trị của SFT\n{'═'*84}")
    print(f"{'nấc':<28}{'EA':>10}{'PA_strict':>12}{'PA_loose':>11}{'no_prog':>10}")
    for _n, _m in [("02_prompt_eng", _m_prev), ("03_sft", metrics)]:
        print(f"{_n:<28}{_m['EA']:>10.4f}{_m['PA_strict']:>12.4f}"
              f"{_m['PA_loose']:>11.4f}{_m['no_program']:>10.4f}")
    print(f"\n  Δ EA        = {metrics['EA'] - _m_prev['EA']:+.4f}")
    print(f"  Δ PA_strict = {metrics['PA_strict'] - _m_prev['PA_strict']:+.4f}")

    for _k in ("ea", "pa_strict"):
        stats.compare_pair(_prev_rows, rows, key=_k,
                           label="NẤC 3 vs NẤC 2 — giá trị của SFT",
                           name_base="02_prompt_eng", name_variant="03_sft")

In [ ]:
save_stage("03_sft", rows, metrics,
           extra={"prompt_level": "engineered", "self_eval": False,
                  "adapter": ADAPTER_DIR, "sft_data": SFT_JSONL if "SFT_JSONL" in dir() else None,
                  "n_sft_records": len(records) if "records" in dir() else None})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f}")

## Kết luận nấc 3

Ba khả năng, và mỗi khả năng có một kết luận khác nhau cho báo cáo:

| Kết quả | Nghĩa là | Viết gì trong bài |
|---|---|---|
| SFT > nấc 2, p < 0.05 | Rejection sampling khắc phục được thất bại SFT của cách cũ | Đóng góp rõ: học được từ train mà không overfit, **không cần API ngoài** |
| SFT ≈ nấc 2 | Model đã bão hoà với dữ liệu tự sinh | Vẫn đáng báo cáo: củng cố luận điểm "inference-time thắng fine-tuning" |
| SFT < nấc 2 | Tự chưng cất làm hẹp phân bố đầu ra | Kiểm tra đường cong loss ở §9; thử `ACCEPT="pa"` cho dữ liệu sạch hơn |

Dù kết quả nào, **nấc 4 và nấc 5 vẫn chạy được trên cả hai model** (gốc và SFT) — hai
notebook sau có cờ `USE_SFT_ADAPTER` để chọn.

**Tiếp theo:** `04_self_evaluation.ipynb`.